In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
source_path = "/Volumes/sep_catalog/sep_schema/grade2/DS_CSV.csv" #the path where my source file exists and i am assigning it to a variable 

In [0]:
source_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_path)
)

source_df.show(5) #reading csv file and storing it to a DataFrame
source_df.printSchema() #displaying schema
print("no of records in source dataframe : ",source_df.count()) #displaying no of records in source dataframe

In [0]:
null_counts = source_df.select(
    [
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in source_df.columns
    ]
)

display(null_counts) #displaying null counts in each column

In [0]:
duplicate_orders = (
    source_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_orders) #displaying duplicate records in order_id

In [0]:
clean_df = (
    source_df

    # Remove unwanted spaces
    .withColumn("first_name", F.trim(F.col("first_name")))
    .withColumn("last_name", F.trim(F.col("last_name")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("payment_method", F.trim(F.col("payment_method")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("email", F.trim(F.col("email")))

    # Standardize text
    .withColumn("first_name", F.initcap(F.col("first_name")))
    .withColumn("last_name", F.initcap(F.col("last_name")))
    .withColumn("city", F.initcap(F.col("city")))

    # Standardize email
    .withColumn("email", F.lower(F.col("email")))

    # Make amount numeric
    .withColumn("amount", F.col("amount").cast("double"))

    # Make quantity integer
    .withColumn("quantity", F.col("quantity").cast("int"))

    # Remove records without business key
    .filter(F.col("order_id").isNotNull())

    # Remove duplicate orders
    .dropDuplicates(["order_id"])
)

clean_df.show(5)
clean_df.printSchema()
print("Rows after transformation:", clean_df.count())

In [0]:
window_spec = Window.orderBy("order_id")

#creating 4 new columns 
initial_silver = (
    clean_df

    .withColumn(
        "order_sk",
        F.row_number().over(window_spec)
    )

    .withColumn(
        "start_date",
        F.current_date()
    )

    .withColumn(
        "end_date",
        F.to_date(F.lit("9999-12-31"))
    )

    .withColumn(
        "is_current",
        F.lit('Y')
    )
)

initial_silver.show(5)

In [0]:
#concate first_name , last_name col as customer_name
initial_silver = initial_silver.withColumn("customer_name", F.concat(F.col("first_name"), F.lit(" "), F.col("last_name")))
initial_silver.show(3)
#after concating drop the old columns
initial_silver = initial_silver.drop("first_name", "last_name")
initial_silver.show(5)

In [0]:
initial_silver_df = initial_silver.select(
    "order_sk",
    "order_id",
    "customer_name",
    "product_id",
    "category",
    "quantity",
    "amount",
    "payment_method",
    "city",
    "email",
    "start_date",
    "end_date",
    "is_current"
)

initial_silver_df.show(5)

In [0]:

silver_path = "/Volumes/sep_catalog/sep_schema/silver_grade/" #here i already have created a volume inside it the delta file created by databricks

(
    initial_silver
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

In [0]:

silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

display(silver_df)

print(
    "Current records:",
    silver_df
    .filter(F.col("is_current") == 'Y') #checking only current records 'Y' = Yes
    .count()
)

###Now comes the incremental load

In [0]:
incremental_path = "/Volumes/sep_catalog/sep_schema/grade2/DS_CSV_incremental_v2.csv" #already file was uploaded to the path with new records and some repeted(updated) records

incremental_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(incremental_path)
)

incremental_df.show(5)
print("no of records day 2:",incremental_df.count())

In [0]:
incremental_clean = (
    incremental_df

    .withColumn("first_name", F.trim(F.col("first_name")))
    .withColumn("last_name", F.trim(F.col("last_name")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("payment_method", F.trim(F.col("payment_method")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("email", F.trim(F.col("email")))

    .withColumn("first_name", F.initcap(F.col("first_name")))
    .withColumn("last_name", F.initcap(F.col("last_name")))
    .withColumn("city", F.initcap(F.col("city")))

    .withColumn("email", F.lower(F.col("email")))

    .withColumn(
        "amount",
        F.col("amount").cast("double")
    )

    .withColumn(
        "quantity",
        F.col("quantity").cast("int")
    )

    .filter(F.col("order_id").isNotNull())

    .dropDuplicates(["order_id"])
)
incremental_clean.show(5)





In [0]:
#concate first_name , last_name col as customer_name
incremental_clean = incremental_clean.withColumn("customer_name", F.concat(F.col("first_name"), F.lit(" "), F.col("last_name")))
incremental_clean.show(3)
#after concating drop the old columns
incremental_clean = incremental_clean.drop("first_name", "last_name")
incremental_clean.show(5)
print("no of records :",incremental_clean.count())

In [0]:
incremental_clean = incremental_clean.select(
    "order_id",
    "product_id",
    "customer_name",
    "category",
    "quantity",
    "amount",
    "payment_method",
    "city",
    "email")

In [0]:
silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

current_silver = (
    silver_df
    .filter(F.col("is_current") == 'Y')
)

print(
    "Current Silver records:",
    current_silver.count()
)

In [0]:
new_records = (
    incremental_clean.alias("src")
    .join(
        current_silver.alias("tgt"),
        F.col("src.order_id") == F.col("tgt.order_id"), #here LEFT JOIN is used because to find only new records(unmatched records from right table)
        "left"
    )
    .filter(
        F.col("tgt.order_id").isNull() #here we are filtering only new records from right table 
    )
    .select("src.*")
)

print("New records:", new_records.count())
new_records.show(5)

In [0]:
changed_records = (
    incremental_clean.alias("src")
    .join(
        current_silver.alias("tgt"),
        F.col("src.order_id") == F.col("tgt.order_id"),
        "inner"
    )
    .filter(
         (F.coalesce(F.col("src.customer_name"), F.lit("")) !=
         F.coalesce(F.col("tgt.customer_name"), F.lit("")))
         |
        (F.coalesce(F.col("src.product_id"), F.lit(-1)) !=
         F.coalesce(F.col("tgt.product_id"), F.lit(-1)))
        |
        (F.coalesce(F.col("src.category"), F.lit("")) !=
         F.coalesce(F.col("tgt.category"), F.lit("")))
        |
        (F.coalesce(F.col("src.quantity"), F.lit(-1)) !=
         F.coalesce(F.col("tgt.quantity"), F.lit(-1)))
        |
        (F.coalesce(F.col("src.amount"), F.lit(-1.0)) !=
         F.coalesce(F.col("tgt.amount"), F.lit(-1.0)))
        |
        (F.coalesce(F.col("src.payment_method"), F.lit("")) !=
         F.coalesce(F.col("tgt.payment_method"), F.lit("")))
        |
        (F.coalesce(F.col("src.city"), F.lit("")) !=
         F.coalesce(F.col("tgt.city"), F.lit("")))
        |
        (F.coalesce(F.col("src.email"), F.lit("")) !=
         F.coalesce(F.col("tgt.email"), F.lit("")))
).select("src.*")
)
print("Changed records:", changed_records.count())
changed_records.show(5)

In [0]:
display(
    changed_records
    .orderBy("order_id")
)

In [0]:
display(
    new_records
    .orderBy("order_id")
)

In [0]:
changed_ids = (
    changed_records
    .select("order_id")
    .distinct()
)

display(changed_ids)

In [0]:
 expired_silver = (
    silver_df.alias("tgt")
    .join(
        changed_ids.alias("chg"),
        F.col("tgt.order_id") == F.col("chg.order_id"),
        "left"
    )
    .withColumn(
        "end_date",
        F.when(
            F.col("chg.order_id").isNotNull() &
            (F.col("tgt.is_current") == 'Y'),
            F.date_sub(F.current_date(), 1)
        )
        .otherwise(F.col("tgt.end_date"))
    )
    .withColumn(
        "is_current",
        F.when(
            F.col("chg.order_id").isNotNull() &
            (F.col("tgt.is_current") == 'Y'),
            F.lit('N')
        )
        .otherwise(F.col("tgt.is_current"))
    )
    .select(
        "tgt.order_sk",
        "tgt.order_id",
        "tgt.customer_name",
        "tgt.product_id",
        "tgt.category",
        "tgt.quantity",
        "tgt.amount",
        "tgt.payment_method",
        "tgt.city",
        "tgt.email",
        "tgt.start_date",
        "end_date",
        "is_current"
    )
)

display(expired_silver)

In [0]:
records_to_insert = (
    new_records
    .unionByName(changed_records)
)
print(
    "Records to insert:",
    records_to_insert.count()
)             #new records 150 and changed records 20 total 170 need to insert

In [0]:
max_sk = (
    silver_df
    .agg(F.max("order_sk").alias("max_sk"))
    .first()["max_sk"]
)

print("Current maximum surrogate key:", max_sk)

In [0]:
new_records_window = Window.orderBy("order_id")

new_versions = (
    records_to_insert

    .withColumn(
        "order_sk",
        F.row_number().over(new_records_window) + max_sk
    )

    .withColumn(
        "start_date",
        F.current_date()
    )

    .withColumn(
        "end_date",
        F.to_date(F.lit("9999-12-31"))
    )

    .withColumn(
        "is_current",
        F.lit('Y')
    )

    .select(
        "order_sk",
        "order_id",
        "customer_name",
        "product_id",
        "category",
        "quantity",
        "amount",
        "payment_method",
        "city",
        "email",
        "start_date",
        "end_date",
        "is_current"
    )
)

In [0]:
display(
    new_versions
    .orderBy("order_id")
)

In [0]:
final_silver_df = (
    expired_silver
    .unionByName(new_versions)
)

print(
    "Final Silver record count:",
    final_silver_df.count()
)

In [0]:
print(
    "Current records:",
    final_silver_df
    .filter(F.col("is_current") == 'Y')
    .count()
)

In [0]:
final_silver_df = final_silver_df.select(

    "order_sk",
    "order_id",
    "customer_name",
    "product_id",
    "category",
    "quantity",
    "amount",
    "payment_method",
    "city",
    "email",
    "start_date",
    "end_date",
    "is_current"
)
final_silver_df.show(50)
final_silver_df.printSchema()

In [0]:
(
    final_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

In [0]:
final_silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

display(final_silver_df)